# Task 2: Predicting severity metrics
Run this notebook to predict the severity (UPDRSM). 

The structure of this notebook is as follows: 
1. Data setup: loading features, metadata, splits
2. Training helper: function to train models and save outputs
3. Mean baseline 
4. Multiple linear regression
5. Random forest
6. 1D-CNN

## 1. Data setup

In [11]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import joblib
from sklearn.dummy import DummyRegressor

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# CHECK PATH WHEN COMPLETE
FEATURES_PATH = ROOT / "data" / "features.csv"
SPLITS_PATH = ROOT / "data" / "splits.json"

# output directory
OUT = ROOT / "presentation_material" / "task2"
(OUT / "predictions").mkdir(parents=True, exist_ok=True)
(OUT / "weights").mkdir(parents=True, exist_ok=True)

# import functions from models 
from src.data import load_metadata
from features import X as features
from src.models.linreg import make_linear_regression
from src.models.rf import make_random_forest

        stride_time  stride_cv  swing_cv  swing_asymmetry  peak_force
GaCo01     1.230322   0.042125  0.054223         0.013902   14.561747
GaCo02     1.103386   0.035667  0.037527         0.001486   15.233429
GaCo03     1.241191   0.049523  0.078598         0.103336   12.754554
GaCo04     1.249965   0.058603  0.064637         0.022956   15.401692
GaCo05     1.051882   0.039561  0.055866         0.013298   12.820625
...             ...        ...       ...              ...         ...
SiPt36     1.065997   0.041058  0.077181         0.066450   14.236129
SiPt37     1.210989   0.056895  0.072182         0.011100   12.766210
SiPt38     1.163753   0.039291  0.049182         0.000858   12.365833
SiPt39     1.067753   0.048137  0.056888         0.065241   21.129340
SiPt40     1.005855   0.045557  0.056307         0.018599   13.568025

[165 rows x 5 columns]
GaCo01    0
GaCo02    0
GaCo03    0
GaCo04    0
GaCo05    0
         ..
SiPt36    1
SiPt37    1
SiPt38    1
SiPt39    1
SiPt40    1
Name

In [12]:
# load data 
features = features.rename_axis("subject").reset_index()
metadata = load_metadata().rename(columns={"ID": "subject"})

# merge metadata and features
patients = features.merge(
    metadata[["subject", "Group", "UPDRSM"]],
    on="subject", how="inner", validate="many_to_one"
)
patients = patients.loc[
    patients["Group"].eq("Pt") & patients["UPDRSM"].notna()
].reset_index(drop=True)

feature_columns = [
    "stride_time", "stride_cv", "swing_cv",
    "swing_asymmetry", "peak_force"
]

# create X and Y 
X = patients[feature_columns]
y = patients["UPDRSM"].astype(float)
print("Patients:", patients["subject"].nunique(), "| Feature matrix:", X.shape)
display(X.head())

Patients: 91 | Feature matrix: (91, 5)


,stride_time,stride_cv,swing_cv,swing_asymmetry,peak_force
0,1.520570,0.076831,0.128824,0.011780,15.297700
1,1.286201,0.054108,0.093835,0.082978,13.863056
2,1.079744,0.054986,0.077635,0.079694,17.089902
3,1.158036,0.046661,0.060210,0.083254,14.352317
4,1.380191,0.088105,0.139013,0.254944,14.187963


In [13]:
# unsure of what this does. need to check
with open(SPLITS_PATH) as f:
    assignment = json.load(f)["task2"]

test_subjects = set(assignment["test"])
folds = []
for fold in assignment["folds"]:
    train_ids, val_ids = set(fold["train"]), set(fold["val"])
    # Essential safety check: keep the held-out test subjects separate.
    assert train_ids.isdisjoint(val_ids | test_subjects)
    assert val_ids.isdisjoint(test_subjects)
    train_idx = np.flatnonzero(patients["subject"].isin(train_ids).to_numpy())
    val_idx = np.flatnonzero(patients["subject"].isin(val_ids).to_numpy())
    folds.append((train_idx, val_idx))

print("Using", len(folds), "saved validation folds; held-out test subjects untouched.")

Using 5 saved validation folds; held-out test subjects untouched.


## 2. Training helper

Function to implement at every model to train the model and save the output. 

In [14]:
# RUN CODE: Define the reusable training helper.
def train_and_save(model_name, make_model):
    for fold_number, (train_idx, val_idx) in enumerate(folds, start=1):
        model = make_model()  # New, unfitted model for this fold.
        model.fit(X.iloc[train_idx], y.iloc[train_idx])  # Train only on training subjects.
        predictions = np.asarray(model.predict(X.iloc[val_idx])).reshape(-1)

        # Save actual values and predictions for the results teammate.
        output = pd.DataFrame({
            "subject": patients.iloc[val_idx]["subject"].to_numpy(),
            "fold": fold_number,
            "actual_UPDRSM": y.iloc[val_idx].to_numpy(),
            "predicted_UPDRSM": predictions
        })
        output.to_csv(
            OUT / "predictions" / f"{model_name}_val_fold{fold_number}.csv",
            index=False
        )
        # Save fitted preprocessing steps and model together.
        joblib.dump(
            model, OUT / "weights" / f"{model_name}_val_fold{fold_number}.joblib"
        )
        print(f"{model_name}: saved validation fold {fold_number} ({len(output)} rows)")

## 3. Mean baseline 
This is used as a reference point. Ideally, our models should perform better than this. We predict the training-fold mean UPDRSM for each validation subject.

In [15]:
train_and_save("mean_baseline", lambda: DummyRegressor(strategy="mean"))

mean_baseline: saved validation fold 1 (15 rows)
mean_baseline: saved validation fold 2 (15 rows)
mean_baseline: saved validation fold 3 (14 rows)
mean_baseline: saved validation fold 4 (14 rows)
mean_baseline: saved validation fold 5 (14 rows)


## 4. Multiple Linear Regression

In [16]:
train_and_save("linear_regression", make_linear_regression)

linear_regression: saved validation fold 1 (15 rows)
linear_regression: saved validation fold 2 (15 rows)
linear_regression: saved validation fold 3 (14 rows)
linear_regression: saved validation fold 4 (14 rows)
linear_regression: saved validation fold 5 (14 rows)


## 5. Random Forest

In [17]:
train_and_save("random_forest", make_random_forest)

random_forest: saved validation fold 1 (15 rows)
random_forest: saved validation fold 2 (15 rows)
random_forest: saved validation fold 3 (14 rows)
random_forest: saved validation fold 4 (14 rows)
random_forest: saved validation fold 5 (14 rows)


## 6. 1D-CNN Regression